# Modulo 3 — Pattern Discovery

L'AI **non** usa pattern predefiniti: un algoritmo non supervisionato (KMeans) raggruppa le configurazioni di mercato ricorrenti sulle feature del Modulo 2. Ogni cluster = un **pattern candidato**, di cui misuriamo frequenza, rendimento medio, prob. rialzo/ribasso, drawdown, profit factor, expectancy e durata.

I pattern vengono validati **out-of-sample** (split temporale train/test) e quelli instabili/overfittati sono **scartati automaticamente**.

**Anti-leakage:** feature causali, scaler+KMeans addestrati solo sul train, etichette (forward return) mai usate come input.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery import PatternDiscovery

## 1. Dati + feature (Moduli 1-2)

In [ ]:
eng = DataEngine()
raw = generate_ohlcv(n=200_000, start='2021-01-01', freq='1min', seed=3)
h1 = eng.to_timeframe(eng.load_dataframe(raw), 'H1')
feats = FeatureEngine().transform(h1, dropna=True)
print('Matrice feature:', feats.shape)

## 2. Scoperta e validazione OOS

Con `horizon=10` valutiamo cosa succede 10 barre dopo l'ingresso. La direzione (long/short) di ogni pattern è decisa dal training; la validazione avviene sul test.

In [ ]:
pd_ = PatternDiscovery(n_clusters=25, horizon=10, train_frac=0.7,
                       min_frequency=0.005, min_profit_factor=1.1, min_count_test=20)
table = pd_.discover(feats)
print(f'Pattern totali: {len(table)} | stabili: {table["stable"].sum()}')
table[['cluster_id','direction','stable','train_expectancy','test_expectancy',
       'test_profit_factor','test_prob_up','test_max_drawdown','test_count']].head(10)

## 3. Solo i pattern stabili

Questi sono i candidati che passano al **Modulo 4 — Strategy Generator**: hanno expectancy positiva sia in-sample sia out-of-sample, frequenza sufficiente e profit factor OOS sopra soglia.

In [ ]:
for p in pd_.stable_patterns():
    print(f'cluster {p.cluster_id:>2} | dir {p.direction:+d} | '
          f'OOS exp {p.test.expectancy:+.5f} | PF {p.test.profit_factor:.2f} | '
          f'P(up) {p.test.prob_up:.2f} | n={p.test.count} | dd {p.test.max_drawdown:.4f}')

## ✅ Modulo 3 verificato

Scoperta non supervisionata, metriche complete per pattern, validazione OOS e scarto automatico degli instabili. Test: `pytest tests/test_pattern_discovery.py` (7/7), incluso un test che **pianta** un pattern mean-reverting e verifica che venga ritrovato.

> Nota: su dati **sintetici GBM** (random walk) è normale trovare pochi o zero pattern stabili — è il comportamento corretto: non esistono pattern reali da scoprire. Su dati di mercato veri il numero sarà diverso.

**Prossimo:** Modulo 4 — Strategy Generator (entry/exit/SL/TP/BE/trailing + filtri di trend/volatilità/orari) a partire dai pattern stabili.